# Day 58 — Datetime & Time Series Analysis
**Month 3 | Week 6 | DatetimeIndex · resample · rolling · dt accessor · Time-based Insights**

---

> **Real-world framing:**
>
> Every business question that matters has a time dimension.
> "Revenue this month vs last month." "7-day rolling average of orders."
> "Which hour of the day drives the most sales?" "Cumulative revenue since January."
>
> Raw data gives you numbers. Time series gives you *trends* — and trends are what
> clients pay for. A dashboard without a time axis is just a table in disguise.
>
> Today you learn to parse, index, resample, and roll over time-series data
> the way professional analysts do — using Pandas' datetime engine directly,
> without any manual loops or date math.

---

**Skills used today:** Python basics, Pandas (all prior Month 3 days), NumPy (Day 57)  
**New today:** `pd.to_datetime` · `DatetimeIndex` · `.dt` accessor · `resample` ·
`rolling` · `shift` · `pct_change` · time-based filtering · period-over-period comparison

**Total: 80 pts + 10★ bonus**

---


---
## 📦 Section 1 — Raw Data (Read Only — Never Modify)

In [1]:
# ── RAW DATA — DO NOT MODIFY BELOW THIS CELL ──────────────────────────────
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# ── ShopEase dataset — 365 days of orders ──────────────────────────────────
rng = np.random.default_rng(seed=42)
n = 500

regions    = ['North', 'South', 'East', 'West']
categories = ['Electronics', 'Clothing', 'Home', 'Sports', 'Books']
segments   = ['Regular', 'Premium', 'VIP']
statuses   = ['Delivered', 'Returned', 'Pending']

raw = pd.DataFrame({
    'order_id'    : [f'ORD{2000+i}' for i in range(n)],
    'order_date'  : pd.to_datetime(
        rng.choice(pd.date_range('2024-01-01', '2024-12-31'), n, replace=True)
    ),
    'region'      : rng.choice(regions,    n, p=[0.30, 0.25, 0.25, 0.20]),
    'category'    : rng.choice(categories, n, p=[0.30, 0.25, 0.20, 0.15, 0.10]),
    'units'       : rng.integers(1, 20, n),
    'unit_price'  : rng.uniform(50, 500, n).round(2),
    'discount_pct': rng.choice([0, 5, 10, 15, 20], n, p=[0.4, 0.25, 0.20, 0.10, 0.05]),
    'segment'     : rng.choice(segments,   n, p=[0.50, 0.35, 0.15]),
    'status'      : rng.choice(statuses,   n, p=[0.70, 0.20, 0.10]),
})
raw['revenue'] = (raw['units'] * raw['unit_price'] * (1 - raw['discount_pct']/100)).round(2)
df = raw.copy()

print(f"Shape: {df.shape}")
print(f"Date range: {df['order_date'].min().date()}  →  {df['order_date'].max().date()}")
print(f"order_date dtype: {df['order_date'].dtype}")
print()
print(df[['order_id','order_date','region','category','revenue']].head(6).to_string(index=False))


Shape: (500, 10)
Date range: 2024-01-02  →  2024-12-31
order_date dtype: datetime64[ns]

order_id order_date region    category  revenue
 ORD2000 2024-02-02   East Electronics  4439.36
 ORD2001 2024-10-10   West    Clothing   916.29
 ORD2002 2024-08-27  South Electronics  2602.92
 ORD2003 2024-06-09  North    Clothing   651.48
 ORD2004 2024-06-07  North        Home   415.46
 ORD2005 2024-11-10   East Electronics   707.85


---
## 📖 Section 2 — Concept Notes

### 1. Parsing Dates — `pd.to_datetime`

Pandas stores dates as `datetime64[ns]` — a 64-bit integer under the hood,
making date arithmetic as fast as number arithmetic.

```python
# Parse from string column
df['order_date'] = pd.to_datetime(df['order_date'])

# Parse with explicit format (faster, safer)
df['order_date'] = pd.to_datetime(df['order_date'], format='%Y-%m-%d')

# Check dtype
df['order_date'].dtype   # datetime64[ns]
```

---

### 2. The `.dt` Accessor — Extract Any Part of a Date

Once a column is `datetime64`, the `.dt` accessor unlocks every date component:

```python
df['year']    = df['order_date'].dt.year
df['month']   = df['order_date'].dt.month
df['month_name'] = df['order_date'].dt.month_name()   # 'January', 'February'...
df['quarter'] = df['order_date'].dt.quarter            # 1, 2, 3, 4
df['week']    = df['order_date'].dt.isocalendar().week # ISO week number
df['day_of_week'] = df['order_date'].dt.day_name()    # 'Monday', 'Tuesday'...
df['day_of_year'] = df['order_date'].dt.day_of_year   # 1–365
```

**Why it matters:** Every time-based groupby starts with `.dt` extraction.
"Revenue by month" = group by `dt.month`. "Best weekday" = group by `dt.day_name()`.

---

### 3. DatetimeIndex & Time-Based Filtering

Setting `order_date` as the index unlocks date-string slicing — the cleanest
way to filter time ranges:

```python
df_ts = df.set_index('order_date').sort_index()

# Filter by year
df_ts['2024']

# Filter by month — use .loc for single month on DataFrame
df_ts.loc['2024-03']

# Filter by range
df_ts['2024-Q1':'2024-Q2']   # Jan–Jun
df_ts['2024-06-01':'2024-08-31']
```

Without a DatetimeIndex, you'd need `.between()` or boolean masks — slower and messier.

---

### 4. `resample` — Group by Time Period

`resample` is the time-series version of `groupby`.
It aggregates data into fixed time buckets:

```python
# Daily total revenue
daily = df_ts['revenue'].resample('D').sum()

# Weekly order count
weekly = df_ts['revenue'].resample('W').count()

# Monthly revenue
monthly = df_ts['revenue'].resample('ME').sum()   # 'ME' = Month End

# Quarterly mean
quarterly = df_ts['revenue'].resample('QE').mean()
```

**Resample frequency aliases:**
| Alias | Meaning |
|-------|---------|
| `'D'` | Daily |
| `'W'` | Weekly (week ending Sunday) |
| `'ME'` | Month End |
| `'QE'` | Quarter End |
| `'YE'` | Year End |
| `'h'` | Hourly |

---

### 5. `rolling` — Moving Window Calculations

`rolling(n)` computes a statistic over the last n periods — the go-to
tool for smoothing noise and spotting trends:

```python
daily_rev = df_ts['revenue'].resample('D').sum()

# 7-day rolling average (smooths day-of-week noise)
rolling_7  = daily_rev.rolling(7).mean()

# 30-day rolling average (smooths monthly seasonality)
rolling_30 = daily_rev.rolling(30).mean()

# The first (n-1) values are NaN — not enough history yet
rolling_7.head(10)   # rows 0–5 will be NaN
```

**`min_periods`:** If you want rolling stats even at the edges:
```python
daily_rev.rolling(7, min_periods=1).mean()   # uses available data
```

---

### 6. `shift` and `pct_change` — Period-over-Period Comparison

```python
monthly_rev = df_ts['revenue'].resample('ME').sum()

# Previous month's revenue (lag by 1)
prev_month = monthly_rev.shift(1)

# Month-over-month growth rate (%)
mom_growth = monthly_rev.pct_change() * 100

# pct_change is equivalent to:
mom_growth = (monthly_rev - monthly_rev.shift(1)) / monthly_rev.shift(1) * 100
```

**`pct_change()` is the analyst's single most-used time series method.**
Every "growth vs last period" calculation uses it.

---

### Common Mistakes → Fixes

| Mistake | Fix |
|---------|-----|
| `resample` errors — not on DatetimeIndex | Set index: `df.set_index('order_date')` first |
| `rolling` on non-sorted dates | Sort before: `.sort_index()` |
| First N rows of rolling are NaN | Expected — use `min_periods=1` if edge values needed |
| Using `'M'` resample alias | Use `'ME'` (Month End) — `'M'` is deprecated in Pandas 2.2+ |
| `df['2024-03']` fails | Column must be DatetimeIndex, not just datetime column |
| `pct_change()` first row is NaN | Expected — no previous period to compare against |


---
## ✏️ Section 3 — Practice Tasks

Total = **80 pts + 10★ bonus**. Never modify raw data. Work on `df = raw.copy()`.

---
### Task A — Date Parsing & Feature Extraction (20 pts)

#### A1 — Parse and Verify (4 pts)
Confirm `order_date` is `datetime64` dtype.  
If it's not already parsed, convert it with `pd.to_datetime`.  
Print `df['order_date'].dtype` and the min/max dates.

#### A2 — Extract Date Components (10 pts)
Using only the `.dt` accessor (no loops), add these columns to `df`:

| Column | Extraction |
|--------|-----------|
| `year` | 4-digit year |
| `month` | Integer month (1–12) |
| `month_name` | Full name ('January'…) |
| `quarter` | Integer quarter (1–4) |
| `day_of_week` | Full weekday name ('Monday'…) |
| `week_number` | ISO week number (use `.dt.isocalendar().week`) |

Print `df[['order_date','year','month','month_name','quarter','day_of_week','week_number']].head(5)`.

#### A3 — Best Day & Best Quarter (6 pts)
Using the extracted columns and groupby:

1. Which **day of week** (e.g. 'Monday') has the highest total revenue?  
   Print: `f"Best weekday: {day} — Total revenue ₹{amount:,.2f}"`

2. Which **quarter** has the highest order count?  
   Print: `f"Busiest quarter: Q{q} with {count} orders"`

Both answers must use computed values — no hardcoding.


In [2]:
# A1 — Parse and Verify (4 pts)
# Check dtype, print it, print min/max dates
print(f"order_date dtype: {df['order_date'].dtype}")
print(f"Min date: {df['order_date'].min().date()}")
print(f"Max date: {df['order_date'].max().date()}")

order_date dtype: datetime64[ns]
Min date: 2024-01-02
Max date: 2024-12-31


In [3]:
# A2 — Extract Date Components (10 pts)
# Add: year, month, month_name, quarter, day_of_week, week_number
# Print first 5 rows of those columns
df['year']        = df['order_date'].dt.year
df['month']       = df['order_date'].dt.month
df['month_name']  = df['order_date'].dt.month_name()
df['quarter']     = df['order_date'].dt.quarter
df['day_of_week'] = df['order_date'].dt.day_name()
df['week_number'] = df['order_date'].dt.isocalendar().week

print(df[['order_date','year','month','month_name','quarter','day_of_week','week_number']].head(5).to_string(index=False))


order_date  year  month month_name  quarter day_of_week  week_number
2024-02-02  2024      2   February        1      Friday            5
2024-10-10  2024     10    October        4    Thursday           41
2024-08-27  2024      8     August        3     Tuesday           35
2024-06-09  2024      6       June        2      Sunday           23
2024-06-07  2024      6       June        2      Friday           23


In [4]:
# A3 — Best Day & Best Quarter (6 pts)
# 1. Best weekday by total revenue — f-string with computed values
# 2. Busiest quarter by order count — f-string with computed values
weekday_rev = df.groupby('day_of_week')['revenue'].sum()
best_day    = weekday_rev.idxmax()
best_day_rev = weekday_rev.max()
print(f"Best weekday: {best_day} — Total revenue ₹{best_day_rev:,.2f}")

# Busiest quarter by order count
q_counts = df.groupby('quarter')['order_id'].count()
best_q   = q_counts.idxmax()
best_q_count = q_counts.max()
print(f"Busiest quarter: Q{best_q} with {best_q_count} orders")


Best weekday: Friday — Total revenue ₹220,007.90
Busiest quarter: Q2 with 130 orders


---
### Task B — DatetimeIndex & Time-Based Filtering (20 pts)

#### B1 — Set DatetimeIndex (4 pts)
Create `df_ts` by setting `order_date` as the index and sorting it.  
Print `df_ts.index.dtype` and `df_ts.shape`.

#### B2 — Filter by Date Range (8 pts)
Using **date-string slicing** on `df_ts` (the DatetimeIndex way — no boolean masks):

1. Filter to **Q1 2024** (Jan–Mar). Print row count and total revenue.
2. Filter to **Jul–Sep 2024** (Q3). Print row count and total revenue.
3. Filter to a **single month**: whichever month had the most orders in the dataset. Print month name, row count, and revenue.

For (3): find the busiest month dynamically — don't hardcode it.

#### B3 — Period Comparison (8 pts)
Compare **H1 (Jan–Jun)** vs **H2 (Jul–Dec)** revenue:

- Filter each half using date-string slicing
- Compute total revenue and order count for each half
- Compute the revenue difference and percentage change from H1 to H2
- Print a clean comparison table:
```
               H1 2024      H2 2024    Change
Orders:            xxx          xxx      +x.x%
Revenue:    ₹xxx,xxx     ₹xxx,xxx     +x.x%
```


In [5]:
# B1 — Set DatetimeIndex (4 pts)
# df_ts = df.set_index('order_date').sort_index()
# Print: index dtype and shape
df_ts = df.set_index('order_date').sort_index()
print(f"Index dtype: {df_ts.index.dtype}")
print(f"Shape: {df_ts.shape}")


Index dtype: datetime64[ns]
Shape: (500, 15)


In [13]:
# B2 — Filter by Date Range (8 pts)
# 1. Q1: df_ts['2024-01':'2024-03']
# 2. Q3: df_ts['2024-07':'2024-09']
# 3. Busiest month: find dynamically, then filter
# B2 — Filter by Date Range (8 pts)
# 1. Q1
q1 = df_ts['2024-01':'2024-03']
print(f"Q1 — rows: {len(q1)}, revenue: ₹{q1['revenue'].sum():,.2f}")

# 2. Q3
q3_data = df_ts['2024-07':'2024-09']
print(f"Q3 — rows: {len(q3_data)}, revenue: ₹{q3_data['revenue'].sum():,.2f}")

# 3. Busiest month
busiest_month_num = df.groupby('month')['order_id'].count().idxmax()
busiest_month_name = df[df['month'] == busiest_month_num]['month_name'].iloc[0]
# Use .loc for single-month DatetimeIndex lookup
m_data = df_ts.loc[f'2024-{busiest_month_num:02d}']
print(f"Busiest month: {busiest_month_name} — rows: {len(m_data)}, revenue: ₹{m_data['revenue'].sum():,.2f}")

Q1 — rows: 122, revenue: ₹326,618.19
Q3 — rows: 122, revenue: ₹331,942.25
Busiest month: June — rows: 53, revenue: ₹142,817.04


In [8]:
# B3 — H1 vs H2 Comparison (8 pts)
# Filter H1 and H2 using date slicing
# Compute revenue, orders, difference, pct_change
# Print comparison table
h1 = df_ts['2024-01':'2024-06']
h2 = df_ts['2024-07':'2024-12']

h1_rev, h2_rev = h1['revenue'].sum(), h2['revenue'].sum()
h1_cnt, h2_cnt = len(h1), len(h2)
rev_diff = h2_rev - h1_rev
rev_pct  = (rev_diff / h1_rev) * 100
cnt_pct  = ((h2_cnt - h1_cnt) / h1_cnt) * 100

print(f"{'':20} {'H1 2024':>14} {'H2 2024':>14} {'Change':>10}")
print("-" * 60)
print(f"{'Orders:':<20} {h1_cnt:>14,} {h2_cnt:>14,} {cnt_pct:>+9.1f}%")
print(f"{'Revenue:':<20} ₹{h1_rev:>13,.2f} ₹{h2_rev:>13,.2f} {rev_pct:>+9.1f}%")


                            H1 2024        H2 2024     Change
------------------------------------------------------------
Orders:                         252            248      -1.6%
Revenue:             ₹   681,626.02 ₹   669,417.61      -1.8%


---
### Task C — Resample & Aggregation (20 pts)

#### C1 — Monthly Revenue Summary (10 pts)
Starting from `df_ts`, resample `revenue` to **month-end** frequency.

Compute a monthly summary DataFrame with these columns:
- `total_revenue` — sum
- `order_count` — count
- `avg_order_value` — mean

Round all values to 2dp.

Print the full 12-month table (it will have 12 rows — one per month).

Then: which month had the highest total revenue? Which had the highest AOV?  
Print both as f-strings using computed values.

#### C2 — Weekly Order Count + Rolling 4-Week Average (10 pts)
Resample to **weekly** frequency: count the number of orders per week.

Then compute a **4-week rolling average** of the weekly order count (use `min_periods=1` so no NaN values appear).

Print the weekly table with both columns:
```
week_end | orders | rolling_4w_avg
```

Then: identify the **single busiest week** (most orders). Print the week ending date and count.

Also identify the **smoothest period** (lowest rolling avg, ignoring the first 3 weeks):
print the week ending date and its 4-week average.


In [9]:
# C1 — Monthly Revenue Summary (10 pts)
# monthly = df_ts['revenue'].resample('ME').agg(...)
# Build summary DataFrame: total_revenue, order_count, avg_order_value
# Print full table
# Print best month by revenue and by AOV
monthly = df_ts['revenue'].resample('ME').agg(
    total_revenue='sum',
    order_count='count',
    avg_order_value='mean'
).round(2)

print(monthly.to_string())
print()

best_rev_month = monthly['total_revenue'].idxmax().strftime('%B %Y')
best_rev       = monthly['total_revenue'].max()
best_aov_month = monthly['avg_order_value'].idxmax().strftime('%B %Y')
best_aov       = monthly['avg_order_value'].max()

print(f"Highest revenue month: {best_rev_month} — ₹{best_rev:,.2f}")
print(f"Highest AOV month:     {best_aov_month} — ₹{best_aov:,.2f}")


            total_revenue  order_count  avg_order_value
order_date                                             
2024-01-31      107620.71           35          3074.88
2024-02-29      126834.32           50          2536.69
2024-03-31       92163.16           37          2490.90
2024-04-30      133486.55           46          2901.88
2024-05-31       78704.24           31          2538.85
2024-06-30      142817.04           53          2694.66
2024-07-31      122938.26           44          2794.05
2024-08-31       89156.23           37          2409.63
2024-09-30      119847.76           41          2923.12
2024-10-31      145068.25           53          2737.14
2024-11-30       80631.17           29          2780.39
2024-12-31      111775.94           44          2540.36

Highest revenue month: October 2024 — ₹145,068.25
Highest AOV month:     January 2024 — ₹3,074.88


In [10]:
# C2 — Weekly Orders + Rolling 4-Week Average (10 pts)
# weekly_orders = df_ts['revenue'].resample('W').count()
# Rename to 'orders', compute rolling_4w_avg with min_periods=1
# Build table, print
# Busiest week and smoothest period
weekly_orders = df_ts['revenue'].resample('W').count().rename('orders')
weekly_orders = weekly_orders.to_frame()
weekly_orders['rolling_4w_avg'] = weekly_orders['orders'].rolling(4, min_periods=1).mean().round(2)

print(weekly_orders.to_string())
print()

busiest_week     = weekly_orders['orders'].idxmax()
busiest_count    = weekly_orders['orders'].max()
print(f"Busiest week ending: {busiest_week.date()} — {busiest_count} orders")

smoothest_week   = weekly_orders['rolling_4w_avg'].iloc[3:].idxmin()
smoothest_avg    = weekly_orders.loc[smoothest_week, 'rolling_4w_avg']
print(f"Smoothest period (week ending): {smoothest_week.date()} — rolling avg {smoothest_avg:.2f}")


            orders  rolling_4w_avg
order_date                        
2024-01-07       7            7.00
2024-01-14      12            9.50
2024-01-21       4            7.67
2024-01-28       6            7.25
2024-02-04      16            9.50
2024-02-11       9            8.75
2024-02-18      14           11.25
2024-02-25      12           12.75
2024-03-03      11           11.50
2024-03-10      10           11.75
2024-03-17       7           10.00
2024-03-24       4            8.00
2024-03-31      10            7.75
2024-04-07       7            7.00
2024-04-14      11            8.00
2024-04-21      14           10.50
2024-04-28      11           10.75
2024-05-05       9           11.25
2024-05-12       4            9.50
2024-05-19      12            9.00
2024-05-26       2            6.75
2024-06-02      12            7.50
2024-06-09      14           10.00
2024-06-16      12           10.00
2024-06-23      14           13.00
2024-06-30       8           12.00
2024-07-07      13  

---
### Task D — Period-over-Period & Trend Analysis (20 pts)

#### D1 — Month-over-Month Growth (10 pts)
From your monthly `total_revenue` series (C1):

1. Use `pct_change()` to compute month-over-month growth rate (multiply by 100, round to 2dp)
2. Use `shift(1)` to get the previous month's revenue alongside the current
3. Build a DataFrame with columns:
   - `revenue` (current month)
   - `prev_revenue` (previous month via shift)
   - `mom_growth_pct` (from pct_change * 100)
4. Print the full table (12 rows — first row will have NaN for prev and growth)
5. Which month had the **highest MoM growth**? Which had the **steepest decline**?  
   Print both, excluding the first row (NaN).

#### D2 — Cumulative Revenue & 30-Day Rolling Average (10 pts)
From your daily revenue series (resample to `'D'`):

1. Compute **cumulative revenue** using `.cumsum()`
2. Compute **30-day rolling average** of daily revenue using `rolling(30, min_periods=1).mean()`
3. Round both to 2dp

Print the first 35 rows to show:
- Days 1–29: rolling avg is based on partial window (min_periods=1 so no NaN)
- Day 30+: full 30-day window kicks in

Then write **one NRA sentence**: what does the 30-day rolling average reveal about business performance that daily revenue alone cannot show?


In [11]:
# D1 — Month-over-Month Growth (10 pts)
# monthly_rev = resample result from C1 (total_revenue column or series)
# mom_growth = monthly_rev.pct_change() * 100
# prev_revenue = monthly_rev.shift(1)
# Build DataFrame: revenue, prev_revenue, mom_growth_pct
# Print table
# Best month growth and worst month decline
monthly_rev = df_ts['revenue'].resample('ME').sum().round(2)
mom = pd.DataFrame({
    'revenue'      : monthly_rev,
    'prev_revenue' : monthly_rev.shift(1),
    'mom_growth_pct': (monthly_rev.pct_change() * 100).round(2)
})
mom.index = mom.index.strftime('%B %Y')
print(mom.to_string())
print()

valid = mom.dropna(subset=['mom_growth_pct'])
best_growth_month  = valid['mom_growth_pct'].idxmax()
best_growth_val    = valid['mom_growth_pct'].max()
worst_decline_month = valid['mom_growth_pct'].idxmin()
worst_decline_val   = valid['mom_growth_pct'].min()
print(f"Highest MoM growth:  {best_growth_month}  ({best_growth_val:+.2f}%)")
print(f"Steepest decline:    {worst_decline_month}  ({worst_decline_val:+.2f}%)")


                  revenue  prev_revenue  mom_growth_pct
order_date                                             
January 2024    107620.71           NaN             NaN
February 2024   126834.32     107620.71           17.85
March 2024       92163.16     126834.32          -27.34
April 2024      133486.55      92163.16           44.84
May 2024         78704.24     133486.55          -41.04
June 2024       142817.04      78704.24           81.46
July 2024       122938.26     142817.04          -13.92
August 2024      89156.23     122938.26          -27.48
September 2024  119847.76      89156.23           34.42
October 2024    145068.25     119847.76           21.04
November 2024    80631.17     145068.25          -44.42
December 2024   111775.94      80631.17           38.63

Highest MoM growth:  June 2024  (+81.46%)
Steepest decline:    November 2024  (-44.42%)


In [12]:
# D2 — Cumulative Revenue + 30-Day Rolling Average (10 pts)
# daily_rev = df_ts['revenue'].resample('D').sum()
# cum_rev = daily_rev.cumsum()
# rolling_30 = daily_rev.rolling(30, min_periods=1).mean().round(2)
# Print first 35 rows
daily_rev  = df_ts['revenue'].resample('D').sum().fillna(0)
cum_rev    = daily_rev.cumsum().round(2)
rolling_30 = daily_rev.rolling(30, min_periods=1).mean().round(2)

result = pd.DataFrame({
    'daily_revenue' : daily_rev,
    'cum_revenue'   : cum_rev,
    'rolling_30d_avg': rolling_30
})
print(result.head(35).to_string())


            daily_revenue  cum_revenue  rolling_30d_avg
order_date                                             
2024-01-02         249.73       249.73           249.73
2024-01-03        9254.97      9504.70          4752.35
2024-01-04        1999.76     11504.46          3834.82
2024-01-05           0.00     11504.46          2876.12
2024-01-06        2688.63     14193.09          2838.62
2024-01-07        9428.03     23621.12          3936.85
2024-01-08        9347.51     32968.63          4709.80
2024-01-09       13219.99     46188.62          5773.58
2024-01-10        4162.92     50351.54          5594.62
2024-01-11           0.00     50351.54          5035.15
2024-01-12        7685.08     58036.62          5276.06
2024-01-13        1982.64     60019.26          5001.61
2024-01-14        5787.82     65807.08          5062.08
2024-01-15           0.00     65807.08          4700.51
2024-01-16        4793.51     70600.59          4706.71
2024-01-17        8346.07     78946.66          

**NRA — 30-day rolling average insight:**

**N:** The 30‑day rolling average of daily revenue rose from ~₹250 to a sustained level of ₹4,500–₹5,000 after the first month.

**R:** This smooths out daily spikes and zero‑revenue days, revealing that the underlying business baseline is roughly ₹4,700/day once the initial ramp‑up period ends. Daily figures alone obscure this stable trend with noise.

**A:** Management should use the 30‑day rolling average as the primary metric for weekly performance reviews, setting alerts when the average drops below ₹4,200 – a signal of a real slowdown needing investigation.

---
## 📊 Section 4 — Scoring Rubric

| Task | Sub-Task | Pts | What Is Checked |
|------|----------|-----|-----------------|
| **A — Date Parsing** | A1 dtype + min/max | 4 | `datetime64` dtype confirmed, min/max dates printed |
| | A2 Six components | 10 | All 6 columns added via `.dt` (no loops), correct types, head(5) printed |
| | A3 Best day + quarter | 6 | Both dynamic (no hardcoding), correct groupby logic, f-string format |
| **B — DatetimeIndex** | B1 Set index | 4 | `df_ts` created, sorted, index dtype printed |
| | B2 Date-string filtering | 8 | All 3 slices use string syntax on DatetimeIndex (not boolean masks), correct counts + revenue |
| | B3 H1 vs H2 | 8 | Both halves filtered correctly, diff and pct computed, table printed |
| **C — Resample** | C1 Monthly summary | 10 | `resample('ME').agg(...)`, 3 columns, 12 rows, best month dynamic |
| | C2 Weekly + rolling | 10 | `resample('W')`, `rolling(4, min_periods=1)`, busiest + smoothest weeks identified |
| **D — Trends** | D1 MoM growth | 10 | `pct_change()` and `shift(1)` used, full table, best/worst months identified dynamically |
| | D2 Cumulative + rolling30 | 10 | `cumsum()` correct, `rolling(30, min_periods=1)`, first 35 rows printed, NRA present |
| **★ Bonus** | See below | 10★ | — |
| **TOTAL** | | **80 + 10★** | |

---

### ⭐ Bonus Task — Seasonality & Weekday Pattern (10★ pts)

#### Part 1 — Revenue Heatmap by Month × Weekday (5★)
Build a pivot table with:
- Rows: month number (1–12)
- Columns: day of week (Monday–Sunday — in correct weekday order)
- Values: mean revenue per cell

```python
# Hint: use pd.Categorical to enforce weekday ordering
day_order = ['Monday','Tuesday','Wednesday','Thursday','Friday','Saturday','Sunday']
df['day_of_week'] = pd.Categorical(df['day_of_week'], categories=day_order, ordered=True)
pivot = df.pivot_table(index='month', columns='day_of_week', values='revenue', aggfunc='mean').round(2)
```

Print the pivot table.

Which (month, weekday) cell has the highest mean revenue?  
Use `np.unravel_index(np.argmax(pivot.values), pivot.shape)` or `.stack().idxmax()`.  
Print: `f"Peak: {month_name} {weekday} — ₹{value:,.2f} avg revenue per order"`

#### Part 2 — Quarter-over-Quarter Growth (5★)
Resample to quarterly revenue (`resample('QE').sum()`).  
Compute QoQ growth rate using `pct_change()`.  
Print a table: Q1, Q2, Q3, Q4 revenue and QoQ growth.  
Which quarter had the highest QoQ growth?


In [14]:
# ⭐ BONUS — Part 1: Revenue Heatmap by Month × Weekday (5★)
# Build pivot: month (rows) × day_of_week (cols, ordered Mon–Sun) × mean revenue
# Print pivot
# Find peak (month, weekday) cell — dynamic
day_order = ['Monday','Tuesday','Wednesday','Thursday','Friday','Saturday','Sunday']
df['day_of_week'] = pd.Categorical(df['day_of_week'], categories=day_order, ordered=True)

# Pivot: month (rows) × weekday (cols) × mean revenue
pivot = df.pivot_table(index='month', columns='day_of_week', values='revenue', aggfunc='mean').round(2)
print("Mean revenue by Month × Weekday:")
print(pivot.to_string())

# Find peak cell
peak_val = pivot.stack().max()
peak_loc = pivot.stack().idxmax()   # (month, weekday)
peak_month_name = pd.to_datetime(f'2024-{peak_loc[0]}-01').strftime('%B')
peak_weekday = peak_loc[1]
print(f"\nPeak: {peak_month_name} {peak_weekday} — ₹{peak_val:,.2f} avg revenue per order")

Mean revenue by Month × Weekday:
day_of_week   Monday  Tuesday  Wednesday  Thursday   Friday  Saturday   Sunday
month                                                                         
1            4088.16  3260.30    2656.58   2292.30  3842.54   2335.64  3466.75
2            2303.98  1884.42    2840.45   2853.78  2645.26   3209.84  1708.15
3             492.22  2960.62    2486.23   1764.04  2554.46   4076.53  2912.35
4            3805.64  2850.76    2023.31   2941.52  2241.82   4214.01  2177.89
5            2128.30  2967.04    2485.51   2446.29  4436.88    847.89  2943.19
6            2008.66  4251.12    2563.37   3688.90  2968.71   2096.94  1436.58
7            3204.75  2180.82    1569.72   3069.68  4275.98   2631.39  2324.16
8            1357.55  1235.98    4481.88   1411.42  2474.90   2698.23  4849.96
9            2778.75  3294.52    2848.85   2964.24  3254.56   3170.87  2057.51
10           2963.41  3340.28    2128.42   2191.82  1831.28   4255.31  2368.47
11           5160.3

In [15]:
# ⭐ BONUS — Part 2: Quarter-over-Quarter Growth (5★)
# quarterly = df_ts['revenue'].resample('QE').sum().round(2)
# qoq = quarterly.pct_change() * 100
# Build table: quarter label, revenue, QoQ growth
# Print, identify highest QoQ growth quarter
quarterly = df_ts['revenue'].resample('QE').sum().round(2)
qoq = quarterly.pct_change() * 100

# Build table
q_table = pd.DataFrame({
    'revenue': quarterly,
    'qoq_growth_pct': qoq.round(2)
})
q_table.index = ['Q1 2024', 'Q2 2024', 'Q3 2024', 'Q4 2024']
print("Quarterly Revenue and QoQ Growth:")
print(q_table.to_string())
print()

# Highest QoQ growth quarter (skip first NaN)
valid_q = q_table.dropna(subset=['qoq_growth_pct'])
best_q = valid_q['qoq_growth_pct'].idxmax()
best_q_val = valid_q['qoq_growth_pct'].max()
print(f"Highest QoQ growth: {best_q} ({best_q_val:+.2f}%)")

Quarterly Revenue and QoQ Growth:
           revenue  qoq_growth_pct
Q1 2024  326618.19             NaN
Q2 2024  355007.83            8.69
Q3 2024  331942.25           -6.50
Q4 2024  337475.36            1.67

Highest QoQ growth: Q2 2024 (+8.69%)


---

### Key Takeaway
Time is the axis that turns data into a story. `resample` collapses 500 rows into 12 monthly snapshots. `rolling` smooths the noise to reveal the trend underneath. `pct_change` turns absolute numbers into the growth narrative clients actually care about. Every dashboard you'll ever build — in Power BI, Tableau, or Streamlit — runs on exactly these three operations before a single visual is drawn.

### Interview Frame
> "When a client asks 'how are we trending?', I set a DatetimeIndex, resample to the right frequency (daily, weekly, monthly), compute a rolling average to strip out noise, and then use pct_change to surface the MoM or QoQ growth. The whole pipeline is five lines of Pandas. I then flag the months with the sharpest declines and the weeks with the highest order spikes — those are the two conversations every client needs to have."
